# Sketch2Shape — basic Google Colab notebook

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order in a fresh runtime.

**Upload a sketch → use the general prompt → generate an image → download PNG.**

For a new sketch, rerun only **Upload**, **Generate**, and **Download**. Model loading is needed once per runtime. All computation and model downloads happen in Colab. This notebook generates a refined 2D image, not a 3D model.

In [ ]:
# Install the image-generation libraries; keep Colab's existing PyTorch.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.35.1", "transformers==4.57.1", "accelerate==1.11.0",
    "peft==0.17.1", "huggingface-hub==0.36.0", "safetensors==0.6.2"])

In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU in Runtime > Change runtime type, then reconnect.")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_scribble",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", controlnet=controlnet,
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print("Model ready. Upload your sketch below.")

### Upload your sketch
Choose any PNG, JPG/JPEG or WebP sketch from your computer. Use clear strokes on a plain background. Upload one image at a time; rerun this cell whenever you want a new sketch.

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
import io

GENERATED = None
CONTROL_IMAGE = None
uploaded = files.upload()
if not uploaded:
    raise ValueError("Choose a sketch image to upload.")
filename, data = next(iter(uploaded.items()))
rgba = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGBA")
image = Image.alpha_composite(Image.new("RGBA", rgba.size, "white"), rgba).convert("RGB")
gray = ImageOps.grayscale(image)
corners = [(0, 0), (gray.width-1, 0), (0, gray.height-1), (gray.width-1, gray.height-1)]
if sum(gray.getpixel(point) for point in corners) / 4 < 128:
    gray = ImageOps.invert(gray)
paper = ImageOps.pad(gray, (512, 512), color=255)
control = ImageOps.invert(paper).point(lambda value: 255 if value > 60 else 0).convert("RGB")
if control.getbbox() is None:
    raise ValueError("The image looks blank. Upload a sketch with visible strokes.")
CONTROL_IMAGE = control
print("Selected:", filename)
display(image)

### General prompt and generation
The prompt below works as a general starting point. You can edit it to describe your object. The model may change details; it does not guarantee an exact reconstruction.

In [ ]:
GENERATED = None
PROMPT = "A realistic studio photograph of the single object shown in the sketch, matching its shape and viewpoint, centered on a plain white background, realistic materials, high detail." #@param {type:"string"}
if CONTROL_IMAGE is None:
    raise RuntimeError("Upload a sketch first.")
result = pipe(
    prompt=PROMPT,
    negative_prompt="multiple objects, duplicate parts, cluttered background, text, watermark, blurry",
    image=CONTROL_IMAGE, num_inference_steps=20, guidance_scale=7.0,
    controlnet_conditioning_scale=1.0,
    generator=torch.Generator(device="cpu").manual_seed(42))
if result.nsfw_content_detected and any(result.nsfw_content_detected):
    raise RuntimeError("The model safety checker flagged this result. Try a different sketch or prompt.")
GENERATED = result.images[0]
GENERATED.save("/content/generated_image.png")
display(GENERATED)

In [ ]:
if GENERATED is None:
    raise RuntimeError("Generate an image successfully before downloading.")
files.download("/content/generated_image.png")